In [ ]:
import time
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp


SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_default_dtype(torch.float64)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

"""
t represents time
Van der Pol oscillator
   x''(t) - mu * (1 - x(t)^2) * x'(t) + x(t) = 0,   t in [0, T]
   x(0)  = x0
   x'(0) = v0
mu, T, x0, v0 are problem parameters (the paper
does not use the Van der Pol equation, so I chose these to give a
moderately nonlinear, well-studied limit-cycle system).
"""
MU = 5.0  # nonlinearity factor
X0 = 1.0  # initial position x(0)
V0 = 0.0  # initial velocity x'(0)

PERIOD_MARGIN = 1.15
ROUGH_PERIOD_FACTOR = 1.614
N_PERIODS_SETTLE = 4

LAYERS = (1, 32, 32, 32, 32, 1)
N_COLLOCATION = 10000
N_INITIAL = 1  # initial collocation point t=0
ADAM_EPOCHS = 5000
ADAM_LR = 1e-3
LBFGS_MAX_ITER = 5000
PRINT_EVERY = 500

In [ ]:
# Finds period of van der pol using reference solution to define T_FINAL
def van_der_pol_rhs(t, y, mu=MU):
    x, v = y
    return [v, mu * (1 - x ** 2) * v - x]

def estimate_vdp_period(mu, x0=X0, v0=V0, rough_factor=ROUGH_PERIOD_FACTOR, n_settle=N_PERIODS_SETTLE):
    rough_T = rough_factor * mu
    t_span = n_settle * max(rough_T, 5.0)
    def event_peak(t, y):
        return y[1]
    event_peak.direction = -1

    sol = solve_ivp(
        lambda t, y: van_der_pol_rhs(t, y, mu=mu),
        [0.0, t_span], [x0, v0],
        events=event_peak, rtol=1e-10, atol=1e-12, max_step=0.05,
    )
    peak_times = sol.t_events[0]
    if len(peak_times) < 3:
        print(f"[period detection] not enough peaks found, falling back to rough estimate {rough_T:.3f}")
        return rough_T
    period = peak_times[-1] - peak_times[-2]
    return period

vdp_period = estimate_vdp_period(MU)
T_FINAL = float(PERIOD_MARGIN * vdp_period)
print(f"Detected VdP period at mu={MU}: {vdp_period:.4f}  ->  T_FINAL = {T_FINAL:.4f}")

In [ ]:
class PINN(nn.Module):
    """
    Fully-connected network approximating u(t) to x(t).
    Input:  t, shape (N, 1)
    Output: x, shape (N, 1)
    """
    def __init__(self, layers=(1, 32, 32, 32, 32, 1)):
        super().__init__()

        self.layers = nn.ModuleList()
        for i in range(len(layers) - 1):
            linear = nn.Linear(layers[i], layers[i + 1])
            # Xavier initialization is the standard choice paired with tanh activations (used in the original PINN paper)
            nn.init.xavier_normal_(linear.weight)
            nn.init.zeros_(linear.bias)
            self.layers.append(linear)

        self.activation = torch.tanh

    def forward(self, t):
        scaled_t = (2.0 * t / T_FINAL) - 1.0
        out = scaled_t
        for layer in self.layers[:-1]:
            out = self.activation(layer(out))
        out = self.layers[-1](out)
        return out

In [ ]:
def physics_residual(model, t, mu=MU):
    """
    Computes the Van der Pol ODE residual f(t) at the given collocation
    points t, using autodiff to obtain x'(t) and x''(t).
    """
    x = model(t)
    # first derivative: dx/dt
    x_t = torch.autograd.grad(
        x, t,
        grad_outputs=torch.ones_like(x),
        create_graph=True,
        retain_graph=True,
    )[0]

    # second derivative: d^2x/dt^2
    x_tt = torch.autograd.grad(
        x_t, t,
        grad_outputs=torch.ones_like(x_t),
        create_graph=True,
        retain_graph=True,
    )[0]
    f = x_tt - mu * (1.0-x**2) * x_t + x # van der pol oscillator equation
    return f

In [ ]:
def sample_collocation_points(n=N_COLLOCATION, t_final=T_FINAL):
    # Randomly sample collocation points in [0, T] for the residual loss
    t_f = np.random.uniform(low=0.0, high=t_final, size=(n, 1))
    t_f = torch.tensor(t_f, dtype=torch.get_default_dtype(), device=device)
    t_f.requires_grad_(True)
    return t_f

def get_initial_condition_data():
    t_ic = torch.zeros((N_INITIAL, 1), dtype=torch.get_default_dtype(), device=device, requires_grad=True)
    x_ic_target = torch.full((N_INITIAL, 1), X0, dtype=torch.get_default_dtype(), device=device)
    v_ic_target = torch.full((N_INITIAL, 1), V0, dtype=torch.get_default_dtype(),device=device)
    return t_ic, x_ic_target, v_ic_target

In [ ]:
# Loss computed using mean squared error
def compute_losses(model, t_ic, x_ic_target, v_ic_target, t_f):
    # IC loss
    x_pred_ic = model(t_ic)
    x_t_pred_ic = torch.autograd.grad(
        x_pred_ic, t_ic,
        grad_outputs=torch.ones_like(x_pred_ic),
        create_graph=True,
        retain_graph=True,
    )[0]

    loss_ic = torch.mean((x_pred_ic - x_ic_target) ** 2) + \
              torch.mean((x_t_pred_ic - v_ic_target) ** 2)

    # Physics residual loss
    f_pred = physics_residual(model, t_f)
    loss_f = torch.mean(f_pred ** 2)

    total_loss = loss_ic + loss_f
    return total_loss, loss_ic, loss_f

In [ ]:
def train(model, adam_epochs=ADAM_EPOCHS, adam_lr=ADAM_LR, lbfgs_max_iter=LBFGS_MAX_ITER, print_every=PRINT_EVERY):
    t_ic, x_ic_target, v_ic_target = get_initial_condition_data()
    t_f = sample_collocation_points()
    history = {"total": [], "ic": [], "f": []}

    optimizer_adam = torch.optim.Adam(model.parameters(), lr=adam_lr)
    for epoch in range(adam_epochs):
        optimizer_adam.zero_grad()
        total_loss, loss_ic, loss_f = compute_losses(model, t_ic, x_ic_target, v_ic_target, t_f)
        total_loss.backward()
        optimizer_adam.step()
        history["total"].append(total_loss.item())
        history["ic"].append(loss_ic.item())
        history["f"].append(loss_f.item())
        if epoch % print_every == 0:
            print(f"[Adam] epoch {epoch:5d} | "
                  f"total {total_loss.item():.3e} | "
                  f"ic {loss_ic.item():.3e} | "
                  f"f {loss_f.item():.3e}")

    optimizer_lbfgs = torch.optim.LBFGS(model.parameters(), max_iter=lbfgs_max_iter, tolerance_grad=1e-9,
                                         tolerance_change=1e-12, history_size=50, line_search_fn="strong_wolfe")
    lbfgs_iter_counter = {"count": 0}
    def closure():
        optimizer_lbfgs.zero_grad()
        total_loss, loss_ic, loss_f = compute_losses(model, t_ic, x_ic_target, v_ic_target, t_f)
        total_loss.backward()
        history["total"].append(total_loss.item())
        history["ic"].append(loss_ic.item())
        history["f"].append(loss_f.item())
        lbfgs_iter_counter["count"] += 1
        if lbfgs_iter_counter["count"] % print_every == 0:
            print(f"[L-BFGS] iter {lbfgs_iter_counter['count']:5d} | "
                  f"total {total_loss.item():.3e} | "
                  f"ic {loss_ic.item():.3e} | "
                  f"f {loss_f.item():.3e}")
        return total_loss
    optimizer_lbfgs.step(closure)

    return history

In [ ]:
model = PINN(layers=LAYERS).to(device)
start_time = time.time()
history = train(model)
training_time = time.time() - start_time
print(f"Training complete in {training_time:.1f}s.")

full_history = history["total"]

In [ ]:
t_eval = np.linspace(0, T_FINAL, 2000)
sol = solve_ivp(van_der_pol_rhs, [0, T_FINAL], [X0, V0], t_eval=t_eval, method="RK45", rtol=1e-10, atol=1e-10)
x_reference = sol.y[0]
v_reference = sol.y[1]

model.eval()
t_test = torch.tensor(t_eval.reshape(-1, 1), dtype=torch.get_default_dtype(), device=device)
with torch.no_grad():
    x_pred = model(t_test).cpu().numpy().flatten()

mse = np.mean((x_pred - x_reference) ** 2)
relative_l2_error = np.linalg.norm(x_pred - x_reference) / np.linalg.norm(x_reference)
final_training_loss = full_history[-1]
n_params = sum(p.numel() for p in model.parameters())
mae = np.mean(np.abs(x_pred - x_reference))
max_error = np.max(np.abs(x_pred - x_reference))
rmse = np.sqrt(mse)

# Mean physics residual on held-out random test points (same recipe as adaptive script)
t_res_test = torch.tensor(
    np.random.uniform(0.0, T_FINAL, size=(2000, 1)),
    dtype=torch.get_default_dtype(), device=device,
)
t_res_test.requires_grad_(True)
f_res_test = physics_residual(model, t_res_test)
mean_physics_residual = np.mean(f_res_test.detach().cpu().numpy() ** 2)

print(f"MSE: {mse:.3e}")
print(f"Relative L2 error: {relative_l2_error:.3e}")
print(f"Final training loss: {final_training_loss:.3e}")
print(f"Mean physics residual (test points): {mean_physics_residual:.3e}")
print(f"Training time: {training_time:.1f}s")
print(f"Trainable parameters: {n_params}")
print(f"MAE: {mae:.3e}")
print(f"Max Error (L_inf): {max_error:.3e}")
print(f"RMSE: {rmse:.3e}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

axes[0, 0].plot(t_eval, x_reference, "b-", linewidth=2, label="Reference (RK45)")
axes[0, 0].plot(t_eval, x_pred, "r--", linewidth=2, label="PINN prediction")
axes[0, 0].set_xlabel("t")
axes[0, 0].set_ylabel("x(t)")
axes[0, 0].set_title("Van der Pol oscillator: PINN vs. reference (vanilla, single-domain)")
axes[0, 0].legend()

axes[0, 1].semilogy(history["total"], label="total loss")
axes[0, 1].semilogy(history["ic"], label="initial condition loss")
axes[0, 1].semilogy(history["f"], label="physics residual loss")
axes[0, 1].set_xlabel("iteration")
axes[0, 1].set_ylabel("loss (log scale)")
axes[0, 1].set_title("Training loss history")
axes[0, 1].legend()

axes[1, 0].plot(t_eval, np.abs(x_pred - x_reference), "k-")
axes[1, 0].set_xlabel("t")
axes[1, 0].set_ylabel("|x_pred - x_reference|")
axes[1, 0].set_title("Absolute error")

x_pred_dot = np.gradient(x_pred, t_eval)
axes[1, 1].plot(x_reference, v_reference, "b-", linewidth=2, label="Reference")
axes[1, 1].plot(x_pred, x_pred_dot, "r--", linewidth=2, label="PINN prediction")
axes[1, 1].set_xlabel("x")
axes[1, 1].set_ylabel("x'")
axes[1, 1].set_title("Phase portrait")
axes[1, 1].legend()

plt.tight_layout()
plt.savefig("pinn_van_der_pol_original_results.png", dpi=150)
plt.show()